# Feature experiments (Mac): one change vs v3, judged against seed noise

Each experiment = `config/{model}/{name}` copied from `ref_version` with ONE change. It reuses the
reference 04b train/test files (symlinks) and runs only 04c + 05c (~20 min each).

- `x_seed*`: same features, different xgboost `random_state` -> spread = noise
- `x_noraw`: drop `X_raw` with an imputed partner
- `x_make`: `dml_make_raw` -> `group_then_ohe:0.01`
- `x_noyrs`: skip `owh_loo_years_cal`

A change is safe if its `diff_vs_ref` is within the spread of the seed runs.
Experiment configs are for local testing only - do not commit `config/*/x_*`.

In [ ]:
model = "car_coll"
ref_version = "v3"
overwrite = True
run = True             # False = only create configs

# name -> create_experiment arguments (one change each)
experiments = {
    "x_seed7":  {"random_state": 7},
    "x_seed11": {"random_state": 11},
    "x_noraw":  {"drop_raw_with_imputed": True},
    "x_make":   {"group_ohe": {"dml_make_raw": 0.01}},
    "x_noyrs":  {"skip": ["owh_loo_years_cal"]},
}

In [ ]:
import sys
from pathlib import Path

project_root = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / 'current.pc').exists())
sys.path.insert(0, str(project_root / 'lib'))
from utils import setup_notebook_environment
from experiments import create_experiment, run_experiment, compare_experiments
project_root = setup_notebook_environment()

In [ ]:
# Create and run (a failed experiment is reported, the others continue)
done, failed = [], {}
for name, kw in experiments.items():
    try:
        create_experiment(project_root, model, name, ref_version=ref_version, overwrite=overwrite, **kw)
        if run:
            run_experiment(project_root, model, name)
        done.append(name)
    except Exception as e:
        failed[name] = repr(e)[:300]
        print(f"  FAILED {name}: {failed[name]}")
print(f"done={done} failed={list(failed)}")

In [ ]:
# Test gini vs ref; seed rows show the noise level
res = compare_experiments(project_root, model, [ref_version] + done, ref=ref_version)
seed_diffs = res.loc[res['version'].str.startswith('x_seed'), 'diff_vs_ref']
noise = max(abs(seed_diffs).max(), 0) if len(seed_diffs) else float('nan')
res['within_seed_noise'] = res['diff_vs_ref'].abs() <= noise
print(f"seed noise (max |diff| of seed runs): {noise:.5f}")
res